# Summary

Query a Vertex AI Search App

In [23]:
import os, sys
import pandas as pd
import json
import time

from google.cloud.discoveryengine_v1 import SearchServiceClient, SearchRequest
from google.api_core.client_options import ClientOptions
from google.cloud import discoveryengine_v1 as discoveryengine


utils_path = "/Users/stephengodfrey/Documents/Workbench/Numantic/utilities/.."
sys.path.insert(0, utils_path)
from utilities.osa_tools.authentication import ApiAuthentication

api_configs = ApiAuthentication(client="Numantic")



In [24]:
class QueryVaiSearch:
    def __init__(self, search_app_id: str, gcp_project: str, gcp_location: str, **kwargs):
        self.gcp_project = gcp_project
        self.gcp_location = gcp_location
        self.search_app_id = search_app_id

        self.serving_config = (
            f"projects/{self.gcp_project}/locations/{self.gcp_location}/"
            f"collections/default_collection/engines/{self.search_app_id}/"
            f"servingConfigs/default_search"
        )

        # Search parameters
        self.return_snippet = True
        self.summary_result_count = 5
        self.include_citations = True
        # self.ignore_adversarial_query = True
        self.ignore_adversarial_query = False
        self.ignore_non_summary_seeking_query = False # Set to False to force summaries
        self.preamble = "Answer the question based on the search results."
        self.model_version = "stable"
        self.page_size = 10

        # New attributes for retrieval
        self.last_summary = ""

        self.__dict__.update(kwargs)
        self.authenticate()

    def authenticate(self):
        client_options = (
            discoveryengine.client_options.ClientOptions(
                api_endpoint=f"{self.gcp_location}-discoveryengine.googleapis.com"
            ) if self.gcp_location != "global" else None
        )
        self.client = discoveryengine.SearchServiceClient(client_options=client_options)

    def search_vertex_ai_app(self, query: str, search_filter: str = ""):
        """
        Performs a search.

        :param query: The natural language search query.
        :param search_filter: SQL-like filter string (e.g., 'country: ANY("USA")')
        """

        content_search_spec = discoveryengine.SearchRequest.ContentSearchSpec(
            snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec.SnippetSpec(
                return_snippet=self.return_snippet
            ),
            summary_spec=discoveryengine.SearchRequest.ContentSearchSpec.SummarySpec(
                summary_result_count=self.summary_result_count,
                include_citations=self.include_citations,
                ignore_adversarial_query=self.ignore_adversarial_query,
                ignore_non_summary_seeking_query=self.ignore_non_summary_seeking_query,
                model_spec=discoveryengine.SearchRequest.ContentSearchSpec.SummarySpec.ModelSpec(
                    version=self.model_version,
                ),
                # Note: ModelPromptSpec preamble is valid in v0.13.12
                model_prompt_spec=discoveryengine.SearchRequest.ContentSearchSpec.SummarySpec.ModelPromptSpec(
                    preamble=self.preamble
                ),
            ),
        )

        request = discoveryengine.SearchRequest(
            serving_config=self.serving_config,
            query=query,
            filter=search_filter, # FEATURE 2: SQL-like WHERE clause
            page_size=self.page_size,
            content_search_spec=content_search_spec,
        )

        # Execute Search
        self.response = self.client.search(request=request)

        # FEATURE 1: Capture the AI RAG Summary
        if hasattr(self.response, 'summary') and self.response.summary:
            self.last_summary = self.response.summary.summary_text
        else:
            self.last_summary = "No summary generated."

        self.extract_structured_data()

    def extract_structured_data(self):
        results_list = []
        for result in self.response.results:
            strut_data_result_dict = {}
            strut_data_result_dict["id"] = result.document.id

            # Map structured data from BigQuery
            for key, value in result.document.struct_data.items():
                strut_data_result_dict[key] = value

            results_list.append(strut_data_result_dict)

        self.struct_data_df = pd.DataFrame(data=results_list)

## Read test data

In [25]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
multi_pas_qs = "multi_passage_answer_questions.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

df_docs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, docs_filename))
df_mpqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, multi_pas_qs))
df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))

In [26]:
df_spqs.head()

,document_index,question,answer
0,0,What do keybullet kin drop?,Keybullet kin drop a key upon death.
1,0,What kind of gun does the bandana bullet kin use?,The bandana bullet kin wields a machine pistol.
2,1,What do the giants look like?,"One giant is burly, grey-skinned, and 20 feet ..."
3,1,What happens on day 2?,"After a few miles of winding tunnel, you emerg..."
4,2,What were the requirements for the project?,The tool had the following requirements:\n- Ch...


## Query

In [27]:
search_app_id = "rag-tests-searchapp-v15"
gcp_project_number = "643773332888"
gcp_location = "global"


## Check the Single passage questions

In [31]:

result_rows = []
for idx in df_spqs.index:

    # Create a query object
    test_querier = QueryVaiSearch(search_app_id=search_app_id,
                                  gcp_project=gcp_project_number,
                                  gcp_location=gcp_location)

    # Query the search app
    try:
        query = df_spqs.loc[idx, "question"]
        test_querier.search_vertex_ai_app(query=query)

        # Put answers is a dictionary
        if len(test_querier.struct_data_df) > 0:
            rag_citations = test_querier.struct_data_df["id"].unique().tolist()
        else:
            rag_citations = []

        sum_dict = dict(query=query,
                        rag_answer=test_querier.last_summary,
                        answer=df_spqs.loc[idx, "answer"],
                        rag_citations=rag_citations,
                        source_document_index="doc_{}.txt".format(df_spqs.loc[idx, "document_index"])
                        )

        result_rows.append(sum_dict)

        time.sleep(5)

    except:
        break

# Put results into a dataframe
df_results = pd.DataFrame(data=result_rows)


In [34]:
dir(test_querier)
test_querier.response
# test_querier.struct_data_df
# test_querier.last_summary
# test_querier.struct_data_df
df_results



,query,rag_answer,answer,rag_citations,source_document_index
0,What do keybullet kin drop?,Keybullet Kin drop a key when killed. [1] If a...,Keybullet kin drop a key upon death.,[0],doc_0.txt
1,What kind of gun does the bandana bullet kin use?,Bandana Bullet Kin use Machine Pistols. [1],The bandana bullet kin wields a machine pistol.,[0],doc_0.txt
2,What do the giants look like?,There are several giants. [1] One giant is des...,"One giant is burly, grey-skinned, and 20 feet ...","[1, 16]",doc_1.txt
3,What happens on day 2?,The provided search results do not contain inf...,"After a few miles of winding tunnel, you emerg...","[17, 16]",doc_1.txt
4,What were the requirements for the project?,"The project, STICI-note, had several key requi...",The tool had the following requirements:\n- Ch...,"[2, 3]",doc_2.txt
5,What data did was used to test the prototype?,The prototype was tested by asking a question ...,Grace Hopper's Wikipedia page and Alan Turing'...,[2],doc_2.txt
6,How do the data storage options compare?,A summary could not be generated for your sear...,For fast start: use SQLite3 and ChromaDB (File...,[],doc_3.txt
7,When was UTF-8 support added for European lang...,UTF-8 encoding for European languages was adde...,UTF-8 support was added for European languages...,[3],doc_3.txt
8,How do I make a button?,"To make a button, you need to import the `mari...",import marimo as mo\n\nbutton = mo.ui.run_butt...,[4],doc_4.txt
9,When might I use caching?,Caching can be used to store expensive interme...,"You might use caching when, for example, your ...","[4, 11]",doc_4.txt


## Add a Filter

In [35]:
testdf = df_results.copy(deep=True)
mask = testdf["rag_citations"].apply(lambda x: len(x)) == 0
testdf[mask].index


Index([6, 13, 19, 23, 31, 36, 37], dtype='int64')

In [37]:
no_ans_idx = testdf[mask].index

result_rows = []
for idx in no_ans_idx:

    # Query the search app
    try:

        # Create a query object
        test_querier = QueryVaiSearch(search_app_id=search_app_id,
                                      gcp_project=gcp_project_number,
                                      gcp_location=gcp_location)

        query = testdf.loc[idx, "query"]

        print(query)

        # Get the source URL for
        doc_index = testdf.loc[idx, "source_document_index"].replace(".txt", "")

        search_filter = 'doc_index: ANY("{}")'.format(doc_index)

        print(search_filter)

        test_querier.search_vertex_ai_app(query=query,
                                          search_filter=search_filter)

        # Put answers is a dictionary
        if len(test_querier.struct_data_df) > 0:
            rag_citations = test_querier.struct_data_df["id"].unique().tolist()
        else:
            rag_citations = []

        sum_dict = dict(query=query,
                        rag_answer=test_querier.last_summary,
                        answer=df_spqs.loc[idx, "answer"],
                        rag_citations=rag_citations,
                        source_document_index="doc_{}.txt".format(df_spqs.loc[idx, "document_index"])
                        )

        result_rows.append(sum_dict)

        time.sleep(5)

    except:
        break

df_results_f = pd.DataFrame(data=result_rows)


How do the data storage options compare?
doc_index: ANY("doc_3")
What kinds of AI carry "systematic risks"?
doc_index: ANY("doc_6")
How do the people who commit atrocious acts and those that are complicit in these acts differ?
doc_index: ANY("doc_9")
For what work did I receive criticism for my reduction of FLOPS?
doc_index: ANY("doc_11")
What languages do the people of Pacifico del Rio speak?
doc_index: ANY("doc_15")
Who wrote 'Divine Rivals'?
doc_index: ANY("doc_18")
Who was resurrected with a group of other murder victims?
doc_index: ANY("doc_18")


In [38]:
df_results_f
# test_querier.response

,query,rag_answer,answer,rag_citations,source_document_index
0,How do the data storage options compare?,A summary could not be generated for your sear...,For fast start: use SQLite3 and ChromaDB (File...,[],doc_3.txt
1,"What kinds of AI carry ""systematic risks""?",A summary could not be generated for your sear...,"For now, general purpose AI models that were t...",[],doc_6.txt
2,How do the people who commit atrocious acts an...,A summary could not be generated for your sear...,The Zone of Interest does not really different...,[],doc_9.txt
3,For what work did I receive criticism for my r...,A summary could not be generated for your sear...,You received criticism for your research on sp...,[],doc_11.txt
4,What languages do the people of Pacifico del R...,A summary could not be generated for your sear...,All of the humans speak a single language that...,[],doc_15.txt
5,Who wrote 'Divine Rivals'?,A summary could not be generated for your sear...,Rebecca Ross wrote 'Divine Rivals'.,[],doc_18.txt
6,Who was resurrected with a group of other murd...,A summary could not be generated for your sear...,Lou was resurrected along with a handful of ot...,[],doc_18.txt
